# 02 — Bronze UCI Ingestion

**VITAL-Flow Medallion Architecture — Bronze Layer**

This notebook ingests the UCI Pima Indians Diabetes CSV, applies zero-as-null correction,
generates synthetic `patient_id`, and writes to the Bronze Parquet landing zone.

**Inputs:** `raw_data/diabetes.csv`

**Outputs:** `bronze/uci/` (Parquet)

In [ ]:
import sys
sys.path.insert(0, "..")
from scripts.utils import load_config, get_spark, get_logger, log_run
config = load_config("../config.yaml")

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timezone
import os

df = pd.read_csv("../" + config["paths"]["raw_uci"], na_values=["?"])
print(f"Raw CSV: {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
# Replace impossible zeros with NaN
for col in config["uci_impossible_zero_columns"]:
    if col in df.columns:
        count = (df[col] == 0).sum()
        df[col] = df[col].replace(0, pd.NA)
        print(f"  Replaced {count} zeros in '{col}'")

# Generate patient_id
df["patient_id"] = "uci_" + pd.Series(range(len(df))).astype(str).str.zfill(6)
df["ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
df.head()

In [ ]:
# Write Bronze Parquet
bronze_path = "../" + config["paths"]["bronze_uci"]
os.makedirs(bronze_path, exist_ok=True)
table = pa.Table.from_pandas(df)
pq.write_to_dataset(table, root_path=bronze_path, existing_data_behavior="overwrite_or_ignore")
print(f"Bronze UCI written to: {bronze_path} ({len(df)} rows)")

In [ ]:
# Register as Delta table (Databricks)
# spark = get_spark()
# bronze = spark.read.parquet(config["paths"]["bronze_uci"])
# bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_uci")
# spark.sql("DESCRIBE TABLE bronze_uci").show()

## ✅ Bronze UCI Ingestion Complete

Written **768 rows** to `bronze/uci/` as Parquet.